In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from tabulate import tabulate
import warnings
import time
import sys, os
from sklearn.model_selection import KFold
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import make_scorer, mean_squared_error
from sklearn.model_selection import train_test_split
from skopt import BayesSearchCV
import keras_tuner as kt
from keras.models import Sequential
from keras.layers import Dense, Input, Normalization
from tensorflow.keras.optimizers import get as get_optimizer
from sklearn.preprocessing import StandardScaler


path = 'datasets/Dataset_2.xlsx'
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
# Read and load dataset
df= pd.read_excel(path, sheet_name="preproc")





2025-04-04 12:04:40.819449: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-04 12:04:40.828996: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1743761080.838648 2449229 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1743761080.841421 2449229 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-04 12:04:40.853305: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

Num GPUs Available:  0


2025-04-04 12:04:41.895042: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


INPUT DATA

In [3]:
X = (df.iloc[:,14:]).values
print(X.shape)

(2905, 530)


OUTPUT DATA

In [4]:
Y = (df.iloc[:,7]).values
print(Y.shape)

(2905,)


**NRMSEPiqr**

In [5]:

def NRMSEPiqr(observed_values, predicted_values):
    # Calculate RMSEP
    rmsep = np.sqrt(np.mean((observed_values - predicted_values) ** 2))
    # Calculate Q1 (25th percentile) and Q3 (75th percentile)
    Q1 = np.percentile(observed_values, 25)
    Q3 = np.percentile(observed_values, 75)

    # Calculate IQR
    IQR = Q3 - Q1

    return rmsep/IQR

NRMSEPiqrscorer = make_scorer(NRMSEPiqr,greater_is_better=False)


In [6]:
X[0].shape

(530,)

Bayseian Optimization

In [7]:

def model_builder(hp):
  
  max_n_layer = 8
  neurons = []
  for i in range(max_n_layer):
    neurons.append(hp.Int(f"n_neurons_{i}", min_value=32, max_value=512, sampling="log"))
  hp_hidden = hp.Int('n_hidden', min_value=1, max_value=8)
  hp_learning_rate = hp.Float('learning_rate',min_value=1e-5, max_value=0.1, sampling="log")
  hp_optimizers = hp.Choice('optimizer', values = ["SGD","RMSprop","Adam"])
  optimizer = get_optimizer(hp_optimizers)
  optimizer.learning_rate = hp_learning_rate

  model = Sequential()
  model.add(Input(shape=X[0].shape))
  for k in range(hp_hidden):
    model.add(keras.layers.Dense(units=neurons[k], activation="relu"))

  model.add(keras.layers.Dense(1))


  model.compile(optimizer=hp_optimizers,
                loss="mean_absolute_error",
                metrics=["MeanSquaredError"])

  return model

Create Tuner

In [8]:
import tempfile
temp_dir = tempfile.TemporaryDirectory()

tuner = kt.BayesianOptimization(model_builder,
                     objective=kt.Objective("val_loss", direction="min"),
                     seed = 777,
                      max_trials = 100, 
                      overwrite=True,
                      directory=temp_dir.name
)


Start Search

In [9]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.25, random_state=777)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

stop_early = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10)

tuner.search(X_train, Y_train, epochs=9999, validation_split=0.2, callbacks=[stop_early], verbose=0)

# Get the optimal hyperparameters
best_hps=tuner.get_best_hyperparameters(num_trials=1)[0]

print(best_hps)


/home/demetra/Desktop/Demetra/venv_tf/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/demetra/Desktop/Demetra/venv_tf/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/demetra/Desktop/Demetra/venv_tf/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/demetra/Desktop/Demetr

Evaluate best model

In [10]:
best_model = tuner.hypermodel.build(best_hps)
best_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_9 (Dense)                 │ (None, 69)             │        36,639 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 118)            │         8,260 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 167)            │        19,873 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 116)            │        19,488 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 114)            │        13,338 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 139)            │        15,985 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 189)            │        26,460 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 1)              │           190 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 140,233 (547.79 KB)

 Trainable params: 140,233 (547.79 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:

trials = 100
random_cv = np.random.seed(777)
#Create seed list
seeds = np.random.choice(trials + 1, size=trials, replace= False)

NRMSE = np.zeros((trials), dtype="float")
MSE = np.zeros((trials), dtype="float")
RMSE = np.zeros((trials), dtype="float")

for i in range(trials):
    optimizer = get_optimizer(best_hps.get('optimizer'))
    optimizer.learning_rate = best_hps.get('learning_rate')
    best_model.compile(optimizer=optimizer, loss="mean_absolute_error",
                metrics=["MeanSquaredError"])

    X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.25, random_state=seeds[i])

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)


    best_model.fit(X_train, Y_train, epochs=9999, validation_split=0.2, callbacks=[stop_early], verbose=0)

    # Make predictions using the testing set
    Y_pred_mlp = best_model.predict(X_test)

    #Compute error
    NRMSE[i] = NRMSEPiqr(Y_test, Y_pred_mlp)
    MSE[i] = mean_squared_error(Y_test, Y_pred_mlp)
    RMSE[i] = tf.sqrt(mean_squared_error(Y_test, Y_pred_mlp))

mean_NRMSEP_mlp = np.mean(NRMSE)
std_nrmsep = np.std(NRMSE)
mean_MSE_mlp = np.mean(MSE)
std_mse = np.std(MSE)
mean_RMSE_mlp = np.mean(RMSE)
std_rmse = np.std(RMSE)
print("Mean NRMSEPiqr: " + str(mean_NRMSEP_mlp))
print("NRMSEPiqr Standard Deviation: " + str(std_nrmsep))
print("Mean MSE: " + str(mean_MSE_mlp))
print("MSE Standard Deviation: " + str(std_mse))
print("Mean RMSE: " + str(mean_RMSE_mlp))
print("RMSE Standard Deviation: " + str(std_rmse))

23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
23/23 